# 🧹 NLP Preprocessing (PT-BR): YouTube Comments
Este notebook utiliza a biblioteca **spaCy** com o modelo de língua portuguesa para realizar lemmatização e tokenização avançada, além de gerar uma WordCloud dos termos mais frequentes.

In [1]:
import pandas as pd
import re
import spacy
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from unidecode import unidecode

# Carregar o modelo do spaCy para Português
nlp = spacy.load('pt_core_news_sm')

# 1. Carregar dados
df = pd.read_parquet('bundesliga_2425_cazetv_chat.parquet')
print(f"Dataset carregado: {len(df):,} comentários.")

Dataset carregado: 246,918 comentários.


## 2. Normalização PT-BR e Futebol
Dicionários focados em gírias de chat e termos esportivos.

In [2]:
pt_br_slang = {
    "vc": "voce",
    "vcs": "voces",
    "pq": "porque",
    "ta": "esta",
    "tava": "estava",
    "mt": "muito",
    "obg": "obrigado",
    "p": "para",
    "q": "que"
}

football_acronyms = {
    "motm": "melhor da partida",
    "var": "arbitro de video",
    "goat": "melhor de todos os tempos",
    "ucl": "champions league"
}

## 3. Pipeline de Limpeza Avançada

In [3]:
def clean_text(text):
    if not isinstance(text, str): return ""
    
    # 1. Lowercasing
    text = text.lower()
    
    # 2. Remover URLs e Menções
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@[\w:]+', '', text)
    
    # 3. Remover caracteres especiais (ASCII)
    text = unidecode(text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 4. Normalizar palavras intensificadas (gooooooal -> goall)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # 5. Slang and Acronym Expansion
    words = text.split()
    words = [pt_br_slang.get(w, w) for w in words]
    words = [football_acronyms.get(w, w) for w in words]
    
    return " ".join(words)

sample = "GOOOOOOAL!!! vc viu? tá mto foda @ Harry, https://caze.tv"
print(f"Exemplo: {clean_text(sample)}")

Exemplo: gooal voce viu esta mto foda harry


## 4. Tokenização e Lemmatização (spaCy)
Utilizamos o motor do spaCy para extrair as raízes gramaticais das palavras em português.

In [4]:
def preprocess_spacy(text):
    cleaned = clean_text(text)
    doc = nlp(cleaned)
    
    # 6. Lemmatization + Stopword removal
    # Filtramos por tokens que não sejam pontuação, espaço ou stopword
    processed = [token.lemma_ for token in doc if not token.is_stop and not token.is_space]
    
    # 7. Token Count Filtering (>5)
    if len(processed) < 5: return None
    
    return " ".join(processed)

# Reduzir dataset para demonstração se for muito lento, ou aplicar em tudo
# Aplicaremos em uma amostra de 50.000 para performance no notebook se necessário
print("Processando comentários via spaCy...")
df['mensagem_limpa'] = df['mensagem'].sample(min(50000, len(df))).apply(preprocess_spacy)
df_final = df.dropna(subset=['mensagem_limpa'])
print(f"Concluído. {len(df_final)} comentários retidos.")

Processando comentários via spaCy...


KeyboardInterrupt: 

## 5. Visualização: WordCloud
Os termos mais recorrentes no chat da Bundesliga na CazéTV.

In [ ]:
all_words = " ".join(df_final['mensagem_limpa'])

wordcloud = WordCloud(
    width=1000, 
    height=600, 
    background_color='white', 
    colormap='viridis', 
    max_words=100
).generate(all_words)

plt.figure(figsize=(15, 8))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Nuvem de Palavras - Bundesliga Live Chat', fontsize=20)
plt.show()